[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/3-5_%EC%84%A0%ED%83%9D%EC%8B%AC%ED%99%94_%EC%8B%9C%EA%B3%84%EC%97%B4%EA%B8%B0%EC%B4%88%EB%AA%A8%EB%8D%B8.ipynb)

# 3-5 · [선택 심화] 고급 시계열 기법 — 유사 궤적, 기초모델, 파인튜닝, 고장 위험 모델

**⏱ 70~90분 · 선택** — 3-4까지 마친 학생, 또는 교수자 시연용입니다. 본 과정의 필수 내용은 아닙니다.

## 🎯 이번 시간의 질문

> 3-2(직선 추세)와 3-3(랜덤 포레스트)보다 **발전된 시계열 기법**을 쓰면 RUL을 더 잘 맞힐까요? 요즘 화제인 **시계열 기초모델**은 우리 데이터로 **추가 학습(파인튜닝)**하면 어떨까요? 고장 기준선을 하나로 고정하지 않고 **고장 지점까지 가변적으로** 예측할 수는 없을까요? 그리고 **어디까지가 한계**일까요?

네 가지 방법을 차례로 시험하고, 3-2·3-3과 **같은 시점, 같은 채점 방식**으로 비교합니다.

| | 방법 | 우리 데이터로 학습? | RUL을 구하는 법 |
|---|---|---|---|
| 3-2 | 추세 외삽 | 기준선만 | 직선이 고장 기준선(1.07 A)에 닿는 시각 |
| 3-3 | 랜덤 포레스트 | 33개 필터로 학습 | RUL을 직접 예측 |
| **1절** | **유사 궤적** | 학습 없이 33개 필터의 궤적과 **비교** | 지금과 비슷했던 과거 필터들이 그 뒤 버틴 시간 |
| **2절** | **기초모델 zero-shot** | 안 함 (다른 분야 시계열로 미리 학습된 모델) | 예측한 미래 전류가 기준선에 닿는 시각 |
| **3절** | **기초모델 파인튜닝** | 33개 필터로 **추가 학습** | 〃 |
| **5절** | **경로 + 고장 위험 모델** | 파인튜닝 모델 + 33개 필터의 고장 기록으로 위험 모델 학습 | 경로 위에서 "아직 고장 안 났을 확률"이 50% 아래로 떨어지는 시각 |

기초모델은 센서마다 따로 예측하는 구조(채널 독립)라, 여기서는 막힘을 가장 잘 보여 주는 **전류**의 미래를 예측합니다. 여러 특징을 함께 쓰는 것은 랜덤 포레스트와 5절의 고장 위험 모델입니다.

**채점 방식** — 평가용 8개 필터를 60초마다(85개 시점) 예측하고, 정비 결정에 중요한 **결정 구간(실제 RUL 300초 이하, 35개 시점)의 평균 오차**를 봅니다. 예측은 300초에서 자릅니다(3-3의 "상한 300초"와 같은 이유).
평가용 8개만으로는 운이 크게 작용하므로, 6절에서는 **41개 필터를 하나씩 평가용으로 빼는 교차검증** 결과(교수자가 같은 코드로 미리 계산, 파인튜닝도 41번)를 함께 봅니다.

⚠️ **3절 파인튜닝은 GPU가 필요합니다.** 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 고른 뒤 시작하세요. GPU가 없거나 설치가 실패해도, 같은 방법으로 미리 계산해 둔 결과를 불러와 끝까지 진행됩니다.

## 0. 모델 설치 (2~5분)

아래 셀은 기초모델 실행 도구(`granite-tsfm`)를 설치합니다. **설치나 다운로드가 실패해도 괜찮습니다.**
설치 후 "세션을 다시 시작하라"는 안내가 나오면 **런타임 → 세션 다시 시작**을 누르고, 이 셀은 건너뛰고 아래 준비 셀부터 실행하세요.

In [ ]:
# ▶ 그대로 실행하세요
%pip install -q granite-tsfm==0.3.9

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/filter_runs.csv",
             "data/filter_windows.csv",
             "data/filter_patchtst_forecasts.npz",
             "data/filter_patchtst_ft_forecasts.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 준비 (▶) — 데이터, 채점 함수, 비교 기준

3-2·3-3의 방법(기준 모델, 추세 외삽, 랜덤 포레스트)도 **같은 85개 시점**에서 다시 채점해 둡니다. 3-3의 결정 구간 오차 75초는 10초마다 모든 시점에서 잰 값이라, 여기서는 조금 다르게 나옵니다.

In [ ]:
# ▶ 그대로 실행하세요
from sklearn.ensemble import RandomForestRegressor

runs = pd.read_csv('data/filter_runs.csv')
windows = pd.read_csv('data/filter_windows.csv')
main = windows[windows['status'] == 'main'].copy()
main['hi'] = main.groupby('run_id')['current_a'].transform(lambda s: s.rolling(3, min_periods=1).mean())
main['rise'] = main['hi'] - main.groupby('run_id')['hi'].transform('first')
main['slope'] = main.groupby('run_id')['hi'].diff(6).fillna(0)
main['start'] = main.groupby('run_id')['current_a'].transform(lambda s: s.iloc[:3].mean())   # 처음 30초 평균 (5절)
train = main[main['split'] == 'train']
test = main[main['split'] == 'test'].copy()
TH = 1.07                                               # 고장 기준선 (3-1·3-2)

# 평가 시점: 평가용 8개 필터의 60초, 120초, … (고장 전까지)
points = [(rid, t) for rid, r in test.groupby('run_id') for t in range(60, r['failure_s'].iloc[0], 60)]
true_rul = np.array([test.loc[test['run_id'] == rid, 'failure_s'].iloc[0] - t for rid, t in points])
now_hi = [test[(test['run_id'] == rid) & (test['time_s'] <= t)]['hi'].iloc[-1] for rid, t in points]

def score(pred):
    """결정 구간(실제 RUL ≤ 300초)의 평균 오차(초). 예측은 300초에서 자른다."""
    pred = np.minimum(np.asarray(pred, dtype=float), 300)
    ok = (true_rul <= 300) & ~np.isnan(pred)
    return round(np.abs(pred[ok] - true_rul[ok]).mean())

# 비교 기준 세 가지 (3-2·3-3과 같은 설정)
cols = ['hi', 'rise', 'slope', 'time_s', 'voltage_v', 'front_c']
rf = RandomForestRegressor(min_samples_leaf=20, random_state=0).fit(train[cols], train['RUL_s'].clip(upper=300))
test['rf'] = rf.predict(test[cols])
life = runs[runs['split'] == 'train']['failure_s'].median()

def trend_rul(run, t, N=10):
    past = run[run['time_s'] <= t].tail(N)
    if len(past) < N:
        return np.nan
    if past['hi'].iloc[-1] >= TH:
        return 0.0
    a, b = np.polyfit(past['time_s'], past['hi'], 1)
    return 1000.0 if a <= 0 else min((TH - b) / a - t, 1000.0)

results = {
    '기준 모델 (수명 중앙값 - t)': score([max(0, life - t) for rid, t in points]),
    '추세 외삽 (3-2)': score([trend_rul(test[test['run_id'] == rid], t) for rid, t in points]),
    '랜덤 포레스트 (3-3)': score([test[(test['run_id'] == rid) & (test['time_s'] == t)]['rf'].iloc[0] for rid, t in points]),
}
print('평가 시점', len(points), '개 · 결정 구간', (true_rul <= 300).sum(), '개')
pd.Series(results, name='결정 구간 평균 오차(초)')

✅ **체크포인트** — 기준 모델 120초, 추세 외삽 107초, 랜덤 포레스트 77초. 이 세 숫자를 기준으로 새 방법들을 봅니다.

---
## 1. 방법 1 · 유사 궤적 RUL — "지금과 비슷했던 과거 필터는 몇 초 더 버텼나?"

정비 현장의 베테랑이 하는 생각과 같습니다. "전류가 이런 모양으로 오르던 필터는 전에 몇 분쯤 더 갔었지."

```
① 지금 필터의 최근 HI 20개(200초)를 잘라 둔다
② 학습용 필터 33개 각각에서, 이 모양과 가장 비슷한 200초 구간을 찾는다 (값 차이가 가장 작은 곳)
③ 가장 비슷했던 필터 8개가 "그 구간 뒤에 실제로 버틴 시간"을 평균한다 → 예측 RUL
```

- 모델을 학습하지 않고 **과거 기록을 그대로 비교**합니다. 기준선도 필요 없습니다. 고장 순간의 전류가 필터마다 달라도(3-1), 비슷한 필터가 **실제로 고장 난 시각**을 쓰기 때문입니다.
- 설정(최근 20개, 8개 필터)은 **학습용 필터 안에서** 필터를 하나씩 빼고 시험해 가장 좋았던 값입니다. 평가용 필터로 고르지 않았습니다.

▶ 위 ①~③을 그대로 옮긴 함수입니다. `show=True`면 찾은 필터 목록도 돌려줍니다.

In [ ]:
# ▶ 그대로 실행하세요
library = [(rid, r['hi'].to_numpy(), r['time_s'].to_numpy(), r['failure_s'].iloc[0]) for rid, r in train.groupby('run_id')]

def similar_rul(run, t, L=20, k=8, show=False):
    """t초까지의 최근 L개 HI와 가장 비슷한 구간을 학습용 필터마다 찾고, 가장 비슷한 k개 필터가 그 뒤 버틴 시간을 평균한다."""
    h = run.loc[run['time_s'] <= t, 'hi'].to_numpy()[-L:]
    found = []
    for rid, hh, ts, F in library:
        if len(hh) < len(h):
            continue
        diff = [np.sqrt(np.mean((hh[j:j + len(h)] - h) ** 2)) for j in range(len(hh) - len(h) + 1)]
        j = int(np.argmin(diff))
        found.append({'필터': rid, '차이(A)': round(diff[j], 4), '비슷한 구간 끝(초)': ts[j + len(h) - 1],
                      '그 뒤 버틴 시간(초)': F - ts[j + len(h) - 1]})
    found = pd.DataFrame(found).sort_values('차이(A)').head(k)
    pred = found['그 뒤 버틴 시간(초)'].mean()
    return (pred, found) if show else pred

평가용 **17번 필터**(F = 473초)가 지금 **300초**째라고 합시다(실제 남은 시간 173초).

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
run = test[test['run_id'] == 17]
pred, found = similar_rul(run, 300, show=True)
print('예측 RUL:', round(pred), '초 / 실제 RUL:', 473 - 300, '초')
found
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 가장 비슷했던 필터 8개와, 그 필터들이 비슷한 구간 뒤에 버틴 시간이 보입니다. 이 평균이 예측입니다(290초, 실제 173초보다 깁니다). 비슷한 모양이었던 필터들도 그 뒤 64초만 버틴 것부터 479초 버틴 것까지 제각각이라 평균이 흔들립니다. 그래도 **왜 이렇게 예측했는지 표로 설명할 수 있다**는 것이 이 방법의 장점입니다.

85개 시점 전체를 채점합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
pred_sim = [similar_rul(test[test['run_id'] == rid], t) for rid, t in points]
results['유사 궤적 (1절)'] = score(pred_sim)
print('유사 궤적 결정 구간 평균 오차:', results['유사 궤적 (1절)'], '초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 89초.

💬 **결과 해설** — 평가용 8개에서는 추세 외삽(107초)보다 낫고 랜덤 포레스트(77초) 다음입니다. **그런데 이 결과는 운이 좋았던 것입니다.** 41개 필터를 하나씩 빼며 시험하면 유사 궤적은 114초로, 추세 외삽(108초)보다도 못합니다(6절). 필터별로 보면 평가용 8개 안에서도 53초(23번)부터 193초(56번)까지 흔들립니다. **평가 필터가 적을 때는 한 번의 결과로 방법의 우열을 정하면 안 된다**는 좋은 예입니다.
그래도 예측 근거(비슷한 필터 목록)를 보여 줄 수 있다는 장점은 남습니다. 과거 고장 이력이 많이 쌓일수록 좋아지는 방법입니다.

---
## 2. 방법 2 · 시계열 기초모델 zero-shot

**시계열 기초모델**은 전력, 교통, 날씨, 센서 등 수많은 분야의 시계열로 미리 학습한 대형 모델입니다. 여기서는 IBM이 2026년 공개한 **PatchTST-FM**(약 3.8억 개 파라미터)을 씁니다.
우리 데이터로 학습하지 않고(**zero-shot**) 과거 전류만 넣으면, 미래 전류를 한 줄이 아니라 **분위수**로 줍니다. 예) 50% 분위수 = 가운데 예측, 90% 분위수 = "높게 나올 수 있는 쪽"(비관적 시나리오).

▶ 모델을 불러옵니다(처음 한 번 약 1.5GB 다운로드). 실패하면 미리 계산해 둔 예측을 씁니다.

In [ ]:
# ▶ 그대로 실행하세요
import math, time
saved = {'zero': np.load('data/filter_patchtst_forecasts.npz'), 'ft': np.load('data/filter_patchtst_ft_forecasts.npz')}
FT_DONE = False            # 3절에서 이 런타임에 파인튜닝하면 True

try:
    import logging, torch
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()     # 설치·로딩 안내 메시지 줄이기
    logging.getLogger('tsfm_public').setLevel(logging.WARNING)
    from tsfm_public import PatchTSTFMForPrediction
    logging.getLogger('tsfm_public.models.patchtst_fm.modeling_patchtst_fm').setLevel(logging.WARNING)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = PatchTSTFMForPrediction.from_pretrained('ibm-granite/granite-timeseries-patchtst-fm-r2',
                                                    revision='b125275f9204d37cbb81fe47b9cf5e08a521e829').to(dev).eval()
    MODE = f'실제 모델 실행 ({dev})'
except Exception as e:
    model, dev = None, 'cpu'
    MODE = f'저장된 예측 결과 사용 ({type(e).__name__})'

def forecast(run, t, kind='zero'):
    """t초까지의 전류로 앞으로 120칸(1,200초)을 예측. kind: 'zero' 또는 'ft'. 결과: (120, 3) = 미래 시점 × [10%, 50%, 90%]."""
    use_model = model is not None and ((kind == 'zero' and not FT_DONE) or (kind == 'ft' and FT_DONE))
    if not use_model:
        return saved[kind][f'r{run["run_id"].iloc[0]}_t{t}']
    history = run.loc[run['time_s'] <= t, 'current_a'].to_numpy()
    x = torch.tensor(history[None, :, None], dtype=torch.float32, device=dev)
    with torch.inference_mode():
        out = model(past_values=x, prediction_length=120)
    q = np.sort(out.quantile_outputs[0, :, :, 0].float().cpu().numpy().T, axis=1)    # (120, 99)
    return q[:, [9, 49, 89]]

def rul_from_path(path, now_hi):
    """예측 경로가 기준선에 처음 닿는 칸 → RUL(초). 1,200초 안에 안 닿으면 1,200."""
    if now_hi >= TH:
        return 0.0
    hit = np.where(path >= TH)[0]
    return float((hit[0] + 1) * 10) if len(hit) else 1200.0

def plot_forecast(run, t, q, title):
    """과거 전류, 예측 범위(10~90%), 기준선, 실제 고장을 한 그림에."""
    F = run['failure_s'].iloc[0]
    future_t = t + np.arange(1, 121) * 10
    past = run[run['time_s'] <= t]
    plt.plot(run['time_s'], run['current_a'], color='lightgray', label='실제 전류 (정답)')
    plt.plot(past['time_s'], past['current_a'], color='black', label='모델에 넣은 과거')
    plt.plot(future_t, q[:, 1], color='C0', label='예측 50%')
    plt.fill_between(future_t, q[:, 0], q[:, 2], alpha=0.3, label='예측 10~90%')
    plt.axhline(TH, color='red', label='고장 기준선')
    plt.axvline(F, color='black', linestyle=':', label='실제 고장')
    plt.xlim(0, max(900, F + 100)); plt.ylim(0.5, 1.3)
    plt.xlabel('경과 시간 (초)'); plt.ylabel('전류 (A)'); plt.title(title)
    plt.legend(fontsize=8)
    plt.show()

print('실행 방식:', MODE)

1절과 같은 **17번 필터, 300초 시점**의 미래 전류를 예측해 그립니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
q_zero = forecast(run, 300, 'zero')
plot_forecast(run, 300, q_zero, 'zero-shot')
print('50% 경로 RUL:', rul_from_path(q_zero[:, 1], now_hi[points.index((17, 300))]), '초 / 실제 173초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 전류가 0.6 → 0.9 A로 올라오는 중인데, 기초모델의 가운데 예측(파랑)은 잠시 오르다가 **다시 0.6~0.7 A로 내려간다**고 봅니다. 그래서 "1,200초 안에 기준선에 안 닿음"이 나왔습니다. 실제로는 계속 올라 473초에 고장 났습니다.
기초모델은 여러 분야의 시계열에서 "튀어 오른 값은 평소 수준으로 **돌아오는**" 패턴을 많이 배웠습니다. 하지만 **필터 막힘은 되돌아가지 않습니다.** 모델은 우리 설비의 이 성질을 모릅니다.

▶ 85개 시점 전체를 예측하고 채점합니다(실제 모델 실행이면 CPU에서 1~2분, GPU에서 수십 초).

In [ ]:
# ▶ 그대로 실행하세요
q_zero_all = [forecast(test[test['run_id'] == rid], t, 'zero') for rid, t in points]
results['기초모델 zero-shot (2절)'] = score([rul_from_path(q[:, 1], h) for q, h in zip(q_zero_all, now_hi)])
zero90 = [rul_from_path(q[:, 2], h) for q, h in zip(q_zero_all, now_hi)]
print('zero-shot 결정 구간 평균 오차 — 50% 경로:', results['기초모델 zero-shot (2절)'], '초 / 90% 경로(비관):', score(zero90), '초')

✅ **체크포인트** — 50% 경로 약 127초, 90% 경로 약 108초.

💬 **결과 해설** — 3.8억 파라미터의 최신 모델이 **직선 추세(107초)보다도 못합니다.** 비관적인 90% 경로를 쓰면 조금 낫지만 기준 모델 수준입니다. **최신·대형 모델이라도 우리 설비 데이터에서 검증하기 전에는 믿을 수 없습니다.**

---
## 3. 방법 3 · 파인튜닝 — 우리 필터 33개로 추가 학습

기초모델이 우리 필터의 성질("막힘은 되돌아가지 않는다")을 모른다면, **가르치면** 됩니다. 이것이 **파인튜닝(fine-tuning)**입니다. 처음부터 학습하는 대신, 이미 학습된 모델을 우리 데이터로 **조금 더** 학습합니다.

```
학습용 필터 33개의 전류 기록을 여러 시점에서 자른다 (필터마다 6번째 기록부터 끝까지 → 예제 1,461개)
   예제 하나 = "이 시점까지의 과거 전류" → "다음 128칸(1,280초)의 실제 전류"
   고장 뒤처럼 실제 값이 없는 칸은 채점에서 뺀다
→ 모델이 예측한 분위수와 실제 전류의 차이가 줄어들도록 모든 파라미터를 조금씩 고친다 (1,461개 예제를 한 번씩 = 1 epoch)
```

- 평가용 8개 필터는 파인튜닝에 **쓰지 않습니다.** 2절과 같은 시점에서 다시 채점합니다.
- T4 GPU에서 약 3~5분 걸립니다. 실행할 때마다 결과가 조금씩 다를 수 있습니다(무작위 순서).

▶ 파인튜닝 함수입니다(읽기만 하세요. 위 그림을 코드로 옮긴 것입니다).

In [ ]:
# ▶ 그대로 실행하세요
def fine_tune(model, train, epochs=1, lr=1e-5, batch_size=16, horizon=128, seed=0):
    """학습용 필터의 전류 기록을 여러 시점에서 잘라 '과거 → 다음 horizon칸'을 맞히도록 추가 학습한다.
    고장 뒤처럼 정답이 없는 칸은 채점(손실)에서 뺀다."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    bb, cfg, dev = model.backbone, model.config, model.device
    CL = cfg.context_length
    samples = []
    for _, r in train.groupby('run_id'):
        s = r['current_a'].to_numpy(np.float32)
        samples += [(s[:c], s[c:c + horizon]) for c in range(6, len(s))]
    lmax = max(len(a) for a, _ in samples) + horizon
    quantiles = torch.tensor(bb.quantile_levels, device=dev).view(1, 1, -1)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.train()
    for _ in range(epochs):
        order = rng.permutation(len(samples))
        for k in range(0, len(order), batch_size):
            batch = [samples[i] for i in order[k:k + batch_size]]
            B = len(batch)
            x = torch.zeros(B, CL)
            pred, miss, pad = (torch.zeros(B, CL, dtype=torch.bool) for _ in range(3))
            for i, (ctx, fut) in enumerate(batch):
                start = CL - len(ctx) - horizon
                m = float(ctx.mean())
                x[i, :start], pad[i, :start] = m, True                          # 앞쪽 빈칸
                x[i, start:start + len(ctx)] = torch.from_numpy(ctx)            # 과거
                x[i, start + len(ctx):] = m
                x[i, start + len(ctx):start + len(ctx) + len(fut)] = torch.from_numpy(fut)   # 맞힐 미래
                pred[i, start + len(ctx):] = True
                miss[i, start + len(ctx) + len(fut):] = True                    # 고장 뒤: 정답 없음
            x, pred, miss, pad = x.to(dev), pred.to(dev), miss.to(dev), pad.to(dev)
            out = model.backbone(x, pred_mask=pred, miss_mask=miss, pad_mask=pad, return_loss=True, context_length=lmax)
            q, target, lm = out.quantile_outputs, out.normed_target.unsqueeze(-1), out.loss_mask
            loss = 2 * torch.abs((target - q) * ((target <= q).float() - quantiles))
            w = lm.reshape(-1, cfg.n_patch, cfg.d_patch).mul(bb.patch_loss_window).reshape(B, -1)
            loss = ((loss * w.unsqueeze(-1)).sum(1) / w.sum(1, keepdim=True).clamp(min=1)).sum(-1).mean()
            loss = loss / math.sqrt(cfg.num_quantile)
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()
    return len(samples)

▶ GPU가 있으면 이 런타임에서 파인튜닝하고, 없으면 같은 방법으로 미리 계산해 둔 결과를 씁니다.

In [ ]:
# ▶ 그대로 실행하세요
if model is not None and dev == 'cuda':
    start = time.time()
    n = fine_tune(model, train)
    FT_DONE = True
    MODE_FT = f'이 런타임에서 파인튜닝함 (예제 {n}개, {time.time() - start:.0f}초)'
else:
    MODE_FT = '저장된 파인튜닝 결과 사용 (GPU 없음 또는 모델 설치 실패)'
print(MODE_FT)

같은 17번 필터, 300초 시점을 파인튜닝한 모델로 예측합니다. 2절의 코드에서 `'zero'`만 `'ft'`로 바꿉니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
q_ft = forecast(run, 300, 'ft')
plot_forecast(run, 300, q_ft, '파인튜닝 후')
print('50% 경로 RUL:', rul_from_path(q_ft[:, 1], now_hi[points.index((17, 300))]), '초 / 실제 173초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 이제 가운데 예측이 **되돌아가지 않고 계속 오릅니다.** 33개 필터로 "막힘은 계속 진행된다"를 배운 것입니다. (저장된 결과 기준: 50% 경로 RUL 340초, 90% 경로 110초 / 실제 173초)

▶ 85개 시점 전체를 채점하고, **전류 예측 자체**가 얼마나 좋아졌는지도 봅니다(다음 300초 동안 50% 예측과 실제 전류의 평균 차이).

In [ ]:
# ▶ 그대로 실행하세요
q_ft_all = [forecast(test[test['run_id'] == rid], t, 'ft') for rid, t in points]
results['기초모델 파인튜닝 (3절)'] = score([rul_from_path(q[:, 1], h) for q, h in zip(q_ft_all, now_hi)])
ft90 = [rul_from_path(q[:, 2], h) for q, h in zip(q_ft_all, now_hi)]

def forecast_error(qs):
    """다음 300초(30칸, 고장 전까지만) 동안 50% 예측과 실제 전류의 평균 차이(A)."""
    errs = []
    for (rid, t), q in zip(points, qs):
        r = test[test['run_id'] == rid]
        fut = r.loc[r['time_s'] > t, 'current_a'].to_numpy()[:30]
        if len(fut):
            errs.append(np.abs(q[:len(fut), 1] - fut).mean())
    return round(np.mean(errs), 3)

print('전류 예측 오차 — zero-shot:', forecast_error(q_zero_all), 'A / 파인튜닝:', forecast_error(q_ft_all), 'A')
print('RUL 결정 구간 평균 오차 — 50% 경로:', results['기초모델 파인튜닝 (3절)'], '초 / 90% 경로:', score(ft90), '초')

✅ **체크포인트** (저장된 결과 기준, 직접 파인튜닝하면 조금 다를 수 있음) — 전류 예측 오차 0.104 → **0.063 A**(40% 감소), RUL 결정 구간 127 → **약 101초**.

💬 **결과 해설** — 파인튜닝 효과는 분명합니다. 전류 예측 오차가 40% 줄었고, RUL 오차도 추세 외삽(107초)보다 좋아졌습니다. 그런데 랜덤 포레스트(77초)에는 여전히 못 미칩니다. 전류를 이렇게 잘 맞히게 됐는데 왜일까요?

---
## 4. 전류를 완벽하게 맞혀도 한계가 있다 — "기준선 방식"의 약점

2·3절은 모두 "미래 전류를 예측 → **기준선 1.07 A**에 닿는 시각 = RUL"로 계산했습니다. 그렇다면 미래 전류를 **완벽하게** 안다면 어떨까요? 정답(실제 미래 HI)을 미리 보는 반칙을 해서 한계를 재 봅니다.

먼저 평가용 필터 8개가 고장 순간에 HI가 얼마였는지 봅니다(3-1의 `groupby().last()`).

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
test.groupby('run_id')['hi'].last().round(3)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 23·52·56·59번 필터는 HI가 **1.07 A에 닿기 전에**(0.98~1.02 A) 고장 났습니다. 이 필터들에는 전류를 완벽하게 예측해도 "기준선에 안 닿음 = 아직 한참 남음"이라는 답이 나옵니다.

▶ 미래 HI를 정확히 안다고 가정한 **완벽한 예측(오라클)**의 RUL 오차입니다.

In [ ]:
# ▶ 그대로 실행하세요
def oracle_rul(run, t):
    """정답인 미래 HI를 미리 보고, 기준선에 처음 닿는 시각으로 RUL을 구한다 (현실에서는 불가능한 '완벽한 예측')."""
    now = run.loc[run['time_s'] <= t, 'hi'].iloc[-1]
    if now >= TH:
        return 0.0
    hit = run[(run['time_s'] > t) & (run['hi'] >= TH)]
    return float(hit['time_s'].iloc[0] - t) if len(hit) else 1200.0

results['(참고) 완벽한 전류 예측 + 기준선'] = score([oracle_rul(test[test['run_id'] == rid], t) for rid, t in points])
print('완벽한 예측 + 기준선 1.07 A의 결정 구간 평균 오차:', results['(참고) 완벽한 전류 예측 + 기준선'], '초')

✅ **체크포인트** — 131초.

💬 **결과 해설** — 전류를 **완벽하게** 맞혀도 131초나 틀립니다. 파인튜닝한 모델(101초)이 오히려 낫게 나온 것은 예측이 조금 높게 나와 기준선에 일찍 닿은 덕분일 뿐입니다. 문제는 예측 실력이 아니라 **기준선 방식 자체**에 있습니다. 3-1에서 본 것처럼 고장 순간의 전류가 필터마다 0.84~1.18 A로 흩어져 있어서, 어떤 하나의 기준선도 모든 필터에 맞지 않습니다(과제 1에서 기준선을 바꿔 확인).
그럼 기준선을 **중앙값이 아닌 다른 방식**으로 정하면 어떨까요? 교수자가 41개 필터 교차검증으로 시험한 결과입니다(`운영도구/analyze_rul_limits.py`).

| 기준선을 정하는 법 (완벽한 전류 예측일 때) | 결정 구간 평균 오차 |
|---|---|
| 학습 필터 고장 순간 HI의 **중앙값** (지금 방식) | 117초 |
| **하위 25%** (보수적으로 낮게) | 113초 |
| 학습 필터 고장값 **전부를 기준선 후보**로 쓰고 RUL의 중앙값 (분포로 다루기) | 121초 |
| **가변 기준선** — 필터 상태(처음 전류, 지금 전류·전압, 기울기)로 "이 필터는 몇 A에서 고장 날지" 예측 | 고장 전류 예측 오차 0.054 A (중앙값 하나 0.060 A와 거의 같음) |

중앙값은 "가운데 추정"일 뿐 정비 기준이 아닙니다. 그렇다고 낮추거나 분포로 다루거나 필터마다 예측해도 크게 나아지지 않습니다. **필터가 몇 A에서 고장 날지는 지금 가진 센서 정보로 미리 알 수 없기 때문입니다.** 반면 랜덤 포레스트(3-3)는 기준선 없이 **"실제로 고장 난 시각"을 직접 배웁니다.** 그래서 작은 모델이 3.8억 파라미터 기초모델을 이깁니다.

그렇다면 기초모델의 좋은 전류 경로는 살리고, 고정 기준선 대신 **고장 지점을 상태에 따라 가변적으로** 정하면 어떨까요? → 5절.

---
## 5. 방법 4 · 고장 지점도 가변적으로 — 경로 + 고장 위험 모델

기준선을 아예 없애고, **"이 상태면 곧 고장 날 확률이 얼마인가"**를 우리 필터 33개의 고장 기록에서 배웁니다. 생존분석(survival analysis)의 **고장 위험(hazard)** 개념입니다.

```
① 고장 위험 모델: 학습용 필터의 모든 시점(10초마다)에서
      [HI, 최근 60초 상승량, 경과 시간, 처음 대비 상승량, HI × 경과 시간, 1 A를 넘은 양] → "30초 안에 고장 났나?(0/1)"
   를 로지스틱 회귀로 학습한다 (특징 여러 개를 함께 쓴다)
② 파인튜닝한 기초모델이 미래 전류 경로를 그린다 (3절)
③ 경로 위 각 칸(10초)마다 고장 확률을 매기고, "아직 고장 안 났을 확률"(생존 확률) = (1 − 고장 확률)을 차례로 곱한다
④ 생존 확률이 50% 아래로 떨어지는 시각 = 예측 RUL
```

고장 지점이 고정된 1.07 A가 아니라 **전류 수준·오르는 속도·경과 시간에 따라 필터마다 달라집니다.** 게다가 RUL 숫자 하나가 아니라 **"언제까지 고장 안 날 확률" 곡선**이 나옵니다.

▶ 고장 위험 모델을 학습하고, 경로 위에서 생존 확률을 계산하는 함수입니다.

In [ ]:
# ▶ 그대로 실행하세요
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def hazard_features(hi, slope, t, start):
    """고장 위험 모델의 입력: HI, 최근 60초 상승량, 경과 시간, 처음 대비 상승량, HI×경과 시간, 1 A를 넘은 양."""
    return np.c_[hi, slope, t, hi - start, hi * t / 1000, np.maximum(hi - 1.0, 0)]

hazard = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
hazard.fit(hazard_features(train['hi'], train['slope'], train['time_s'], train['start']), (train['RUL_s'] <= 30).astype(int))

def survival_curve(run, t, path_current):
    """미래 전류 경로 위에서 10초마다 '아직 고장 안 났을 확률'을 계산한다. 결과: (미래 시각, 생존 확률)."""
    past = run[run['time_s'] <= t]
    cur = np.r_[past['current_a'].to_numpy()[-2:], path_current]
    path_hi = np.convolve(cur, np.ones(3) / 3, 'valid')                 # HI처럼 30초 이동평균
    hist = np.r_[past['hi'].to_numpy()[-6:], path_hi]
    slope = hist[6:] - hist[:-6]                                        # 최근 60초 상승량
    future_t = t + np.arange(1, 121) * 10
    p30 = hazard.predict_proba(hazard_features(path_hi, slope, future_t, past['start'].iloc[0]))[:, 1]
    h10 = 1 - (1 - p30) ** (1 / 3)                                      # 30초 안 고장 확률 → 10초당 고장 확률
    return future_t, np.cumprod(1 - h10)

def hazard_rul(run, t, path_current):
    """생존 확률이 50% 아래로 떨어지는 시각 − 지금 = 예측 RUL (1,200초 안에 안 떨어지면 1,200)."""
    future_t, S = survival_curve(run, t, path_current)
    k = np.where(S <= 0.5)[0]
    return float(future_t[k[0]] - t) if len(k) else 1200.0

같은 **17번 필터, 300초 시점**의 파인튜닝 경로(3절의 `q_ft`) 위에서 생존 확률 곡선을 그립니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
future_t, S = survival_curve(run, 300, q_ft[:, 1])
plt.plot(future_t, S)
plt.axhline(0.5, color='gray', linestyle='--', label='생존 확률 50%')
plt.axvline(473, color='black', linestyle=':', label='실제 고장')
plt.xlabel('경과 시간 (초)')
plt.ylabel('아직 고장 안 났을 확률')
plt.legend()
plt.show()
print('예측 RUL:', hazard_rul(run, 300, q_ft[:, 1]), '초 / 실제 173초')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 생존 확률이 경로를 따라 서서히 떨어지다가 약 540초에서 50% 아래로 내려갑니다 → 예측 RUL 240초(실제 173초). 같은 경로에 고정 기준선을 쓴 3절(340초)보다 실제에 가깝습니다. 실제 고장 시각(473초)의 생존 확률은 약 0.7로, 모델은 "이때까지 고장 났을 확률 30%"라고 본 셈입니다. **이 곡선을 그대로 교체 결정에 쓸 수 있습니다** — 예) "준비 시간(30초) 안에 고장 날 확률이 10%를 넘으면 교체"처럼, 3-4의 비용 기준으로 확률 기준을 정합니다.

▶ 85개 시점 전체를 채점합니다. 비교를 위해 3-2의 **직선 추세 경로**에 같은 고장 위험 모델을 붙인 결과도 봅니다.

In [ ]:
# ▶ 그대로 실행하세요
def trend_path(run, t, N=10):
    """최근 N개 HI에 맞춘 직선을 앞으로 120칸 연장한 경로."""
    past = run[run['time_s'] <= t].tail(N)
    a, b = np.polyfit(past['time_s'], past['hi'], 1)
    return a * (t + np.arange(1, 121) * 10) + b

results['파인튜닝 경로 + 고장 위험 모델 (5절)'] = score(
    [hazard_rul(test[test['run_id'] == rid], t, q[:, 1]) for (rid, t), q in zip(points, q_ft_all)])
results['추세 경로 + 고장 위험 모델'] = score(
    [hazard_rul(test[test['run_id'] == rid], t, trend_path(test[test['run_id'] == rid], t)) for rid, t in points])
print('파인튜닝 경로 + 고장 위험 모델:', results['파인튜닝 경로 + 고장 위험 모델 (5절)'], '초 (고정 기준선일 때', results['기초모델 파인튜닝 (3절)'], '초)')
print('추세 경로 + 고장 위험 모델   :', results['추세 경로 + 고장 위험 모델'], '초 (고정 기준선일 때', results['추세 외삽 (3-2)'], '초)')

✅ **체크포인트** — 평가용 8개 기준 파인튜닝 경로 + 고장 위험 모델 81초(고정 기준선 101초), 추세 경로 + 고장 위험 모델 85초(고정 기준선 107초).

💬 **결과 해설** — 고장 지점을 가변적으로 보자 두 경로 모두 좋아졌습니다(파인튜닝 경로 101 → 81초, 추세 경로 107 → 85초). 기초모델이 그린 좋은 경로를 고정 기준선이 망치고 있었다는 4절의 진단이 맞았던 것입니다. 41개 교차검증에서도 파인튜닝 경로 103 → 88초, 추세 경로 108 → 91초로 효과가 유지됩니다(6절). 다만 평가용 8개에서 랜덤 포레스트(77초)와 비슷해 보인 것은 운이 섞인 결과로, 41개 교차검증에서는 랜덤 포레스트(74초)가 여전히 앞섭니다.

---
## 6. 한눈에 비교 (▶) — 평가용 8개 vs 41개 교차검증

41개 교차검증 열은 교수자가 같은 코드로 필터 41개를 하나씩 빼며 계산한 값입니다(파인튜닝도 필터마다 새로 41번, `운영도구/lofo_rul_benchmark.py`). 평가 시점이 179개로 늘어 **평가용 8개보다 훨씬 믿을 만합니다.**

In [ ]:
# ▶ 그대로 실행하세요
lofo = {'랜덤 포레스트 (3-3)': 74, '파인튜닝 경로 + 고장 위험 모델 (5절)': 88, '추세 경로 + 고장 위험 모델': 91,
        '기초모델 파인튜닝 (3절)': 103, '추세 외삽 (3-2)': 108, '유사 궤적 (1절)': 114,
        '(참고) 완벽한 전류 예측 + 기준선': 117, '기초모델 zero-shot (2절)': 132, '기준 모델 (수명 중앙값 - t)': 136}   # 운영도구/lofo_rul_benchmark.json
table = pd.DataFrame({'평가용 8개': pd.Series(results), '41개 교차검증': pd.Series(lofo)}).sort_values('41개 교차검증')
plt.figure(figsize=(8, 4.5))
plt.barh(table.index, table['41개 교차검증'], color=['lightgray' if '참고' in i else 'C2' for i in table.index])
plt.gca().invert_yaxis()
plt.xlabel('결정 구간 평균 오차 (초, 작을수록 좋음)')
plt.title('41개 필터 교차검증 · 179개 시점')
plt.show()
table

💬 **결과 해설** — 41개 교차검증에서 보이는 것:
- **랜덤 포레스트(74초)가 여전히 가장 정확합니다.** 두 평가 모두에서 1등인 유일한 방법입니다.
- **고장 위험 모델의 효과는 진짜입니다.** 파인튜닝 경로 103 → 88초, 추세 경로 108 → 91초. 다만 두 경로의 차이(3초)는 교차검증의 흔들림(±4초 정도) 안이라, "기초모델 경로가 더 낫다"고 단정하지 않습니다.
- **유사 궤적은 평가용 8개에서 운이 좋았습니다**(89초 → 114초, 추세 외삽보다 못함).
- zero-shot 기초모델(132초)은 센서를 보지 않는 기준 모델(136초)과 비슷한 수준입니다.

---
## 7. 이 정도가 최선일까? — 한계는 모델이 아니라 정보

방법을 바꿔도 75~90초 근처에서 멈춥니다. 정말 한계인지, 교수자가 41개 교차검증으로 세 가지를 확인했습니다(`운영도구/analyze_rul_limits.py`, 랜덤 포레스트 기준).

| 확인한 것 | 결과 (결정 구간 평균 오차) | 뜻 |
|---|---|---|
| **데이터를 더 모으면?** 학습 필터 10 → 20 → 30 → 40개 | 83 → 77 → 77 → 74초 | 같은 종류의 데이터를 늘려도 거의 제자리 |
| **특징을 더 넣으면?** 6개 → 22개(온도 4곳, 전압, 여러 구간 기울기, 전력 등) | 74 → 77초 | 온도는 운전 조건, 막힘 정보는 이미 전류·전압에 |
| **더 촘촘한 기록을 쓰면?** + 1초 기록의 튐·흔들림 5개 | 74 → 75초 | 10초 평균으로 지운 신호에도 전조가 없음 |
| **남은 오차는 어디서?** 실제 RUL 구간별 | 0~100초 **99초**(평균 97초 길게 예측) · 100~200초 60초 · 200~300초 59초 | 고장 직전 100초를 못 맞힘 |

1초 기록의 전류 튐도 고장이 다가올수록 서서히 커질 뿐입니다(300초 이상 전 0.046 A → 고장 30초 전 0.085 A). **막힌 필터가 마지막에 언제 멈출지는 지금 기록된 신호(전류·전압·온도)에 담겨 있지 않습니다.** 고장 100~300초 전은 60초 안팎으로 맞히지만, 마지막 순간은 이 센서들로는 알 수 없는 우연에 가깝습니다.

### PHM이 어려운 진짜 이유 — 데이터를 얻기 어렵다

- **고장 데이터가 드물다** — 현장 설비는 거의 고장 나지 않습니다(1-1: 디스크 90일 고장률 0.33%). 고장까지 기록한 사례를 모으는 데 몇 년이 걸립니다.
- **고장까지 실험하려면 비싸다** — 이 필터 데이터도 가루를 섞어 막힘을 **앞당긴 실험실** 데이터이고, 그렇게 해도 60회입니다. 실제 라인에서 필터를 고장 날 때까지 쓰는 실험은 생산 손실 때문에 하기 어렵습니다.
- **필요한 정보가 기록되지 않는다** — 필터 막힘을 직접 보여 주는 것은 필터 앞뒤 압력차(ΔP)나 배출량이지만, 이 실험은 모터 전류로 간접 측정했습니다. 오늘 본 것처럼 같은 종류의 데이터를 늘리는 것으로는 부족하고, **정보가 있는 데이터**(맞는 센서, 정비·고장 기록)를 설계해서 모아야 합니다.

그래서 현장 PHM의 순서는 보통 이렇습니다: ① 막힘·마모를 직접 보는 센서와 정비 기록부터 갖춘다 → ② 단순한 모델(규칙, 랜덤 포레스트)로 시작해 기준 모델과 비교한다 → ③ 고장 이력이 쌓이면 확률 예측(5절의 생존 곡선)과 비용(3-4)으로 교체 시점을 정한다 → ④ 고장·교체가 생길 때마다 다시 학습한다. 대형 모델은 그다음입니다.

💬 **이번 시간의 교훈**

1. **최신 모델도 우리 데이터로 검증하기 전에는 믿지 않는다.** zero-shot 기초모델은 직선 추세보다도 못했습니다.
2. **파인튜닝은 효과가 크다.** 33개 필터만으로도 전류 예측 오차가 40% 줄었습니다.
3. **무엇을 예측해서 결정에 쓰느냐가 모델 크기보다 중요하다.** 고정 기준선은 완벽한 예측도 113초 이상 틀립니다. 고장 지점을 가변적으로 보는 위험 모델이나, RUL을 직접 배우는 랜덤 포레스트가 더 정확했습니다.
4. **평가 필터 8개의 결과는 믿기 어렵다.** 유사 궤적은 8개에서 2등, 41개 교차검증에서는 하위권이었습니다.
5. **한계는 모델이 아니라 정보다.** 데이터·특징·모델을 늘려도 75초 근처에서 멈춥니다. 더 나아지려면 정보가 있는 데이터를 모아야 하고, 그 전까지는 확률과 비용으로 결정하는 것이 최선입니다.

---
## 8. 과제 (15분)

### 📝 과제 1 · 기준선을 바꾸면 한계가 줄어들까?
4절의 `oracle_rul`은 전역 변수 `TH`를 씁니다. `TH`를 **0.95, 1.0**으로 바꿔 완벽한 예측의 결정 구간 오차를 구하세요. 랜덤 포레스트(77초)를 이길 수 있나요? 끝나면 `TH = 1.07`로 되돌립니다. (4절 표의 41개 교차검증 결과와도 비교해 보세요.)

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for TH in [0.95, 1.0]: print(TH, score([oracle_rul(test[test['run_id'] == rid], t) for rid, t in points]))


✏️ **나의 답:**

### 📝 과제 2 · 비관적 시나리오로 경보 만들기 — zero-shot vs 파인튜닝
"**90% 경로의 RUL이 60초 이하**가 되면 교체"라는 규칙을 평가용 8개 필터에 적용하세요. 필터마다 첫 경보 시각과 그때 실제로 남은 시간을 표로 만들고, 3-4처럼 **30초 이상 남기고 경보한** 필터가 몇 개인지 zero-shot과 파인튜닝을 비교하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: res = pd.DataFrame(points, columns=['run_id', 't']); res['zero90'] = zero90; res['ft90'] = ft90
# alarm = res[res['zero90'] <= 60].groupby('run_id')['t'].min()  → 남은 시간 = F − 경보 시각


✏️ **나의 답:**

### 📝 과제 3 · 생각해 보기
기초모델을 현장 CBM에 도입하자는 제안이 왔습니다. 오늘 결과를 근거로 **찬성 이유 1개, 우려 1개, 도입 전에 꼭 할 일 1개**를 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 방법 | 평가용 8개 | 41개 교차검증 | 한 줄 평가 |
|---|---|---|---|
| 랜덤 포레스트 (3-3) | 77초 | **74초** | RUL을 직접 배운 작은 모델이 가장 정확 |
| 파인튜닝 경로 + 고장 위험 모델 (5절) | 81초 | 88초 | 고장 지점을 가변적으로, 생존 확률 곡선을 준다 |
| 추세 경로 + 고장 위험 모델 | 85초 | 91초 | 경로가 단순해도 위험 모델의 효과가 크다 |
| 기초모델 파인튜닝 (3절) | 약 101초 | 103초 | 전류 예측 40% 개선, 고정 기준선이 발목 |
| 추세 외삽 (3-2) | 107초 | 108초 | 가장 단순한 고정 기준선 방식 |
| 유사 궤적 (1절) | 89초 | 114초 | 평가용 8개에서는 운이 좋았다 |
| (참고) 완벽한 전류 예측 + 고정 기준선 | 131초 | 117초 | 고정 기준선 방식의 한계 |
| 기초모델 zero-shot (2절) | 약 127초 | 132초 | 막힘이 "되돌아간다"고 예측 |
| 기준 모델 (수명 중앙값 − t) | 120초 | 136초 | 센서를 보지 않는 비교 기준 |

- 새 기법은 **같은 데이터, 같은 채점, 충분한 평가 설비 수**로 기준 모델·기존 방법과 비교해야 가치를 알 수 있다.
- 파인튜닝은 대형 모델을 우리 설비에 맞추는 강력한 방법이지만, **문제를 어떻게 설정했는지**(고정 기준선인지, 고장 위험·RUL 직접인지)가 결과를 더 크게 좌우한다.
- 이 데이터의 한계(약 75초)는 모델이 아니라 **정보**에서 온다. PHM의 가장 큰 벽은 **정보가 있는 고장 데이터를 얻는 것**이다.